# Model Development: Risk Clustering and Price Movement Classification

1. **Clustering:** K-Means groups NASDAQ-100 companies by return, volatility and volume statistics. The company closest to the centre of the safest cluster is used as the representative.
2. **Classification:** XGBoost predicts the six-month price movement class from technical and macroeconomic features. It is evaluated with cross-validation and a rolling backtest against a majority-class baseline.

Data is read from the project's PostgreSQL warehouse. Connection details come from environment variables (see `.env.example`).

## 🔍  Problem Statement

In [ ]:
# The problem we've noticed and focused on is the vagueness of starting with investing in stocks.
# To solve this issue, we wanted to create a chatbot for users to ask advice on which company
# is the safest to buy stocks from/sell stocks of at that point in time and the near future.
# To achieve this, we will use a clustering model to find the centroid of the safest companies
# and classification to determine which companies will have very positive to very negative change.
# We aim to make the stock market more accessible to people who do not have the time or
# knowledge to analyse the market.

## Getting Rid of Warnings

In [ ]:
import warnings
from pandas.errors import SettingWithCopyWarning

# Ignore sklearn deprecation warnings about _check_n_features/_check_feature_names
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
# Ignore XGBoost warning about use_label_encoder
warnings.filterwarnings("ignore", message=".*use_label_encoder.*")
# Ignore silhouette_score warnings
warnings.filterwarnings("ignore", message=".*silhouette.*")
# Ignore pandas SettingWithCopyWarning
warnings.filterwarnings("ignore", category=SettingWithCopyWarning)
# Ignore all other UserWarnings
warnings.filterwarnings("ignore", category=UserWarning)


# Clustering Model

##  🗄️ Step 1: Load and Explore Data

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from stock_chatbot.config import get_db_url
 
# Connection details come from environment variables (see .env.example)
db_url = get_db_url()
 
query = '''
SELECT *
FROM stock_model_features;
'''
 
engine = create_engine(db_url, echo=True)
df_alch = pd.read_sql_query(query, engine)

In [ ]:
df_alch.head()

In [ ]:
df_alch.to_csv('stock_model_features.csv', index=False)

## 📊 Step 2: Preprocessing

#### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances_argmin_min
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import plotly.express as px

#### Loading The Data

In [ ]:
df = pd.read_csv('stock_model_features.csv', parse_dates=True, index_col='date_value')
df.head()

In [ ]:
df.columns

#### Preprocessing

In [ ]:
df = df.sort_index(ascending=True)
cutoff_date = df.index.max() - pd.DateOffset(months=6)
df_6m = df[df.index >= cutoff_date].copy()
df_6m = df_6m[['ticker', 'close_value', 'open_value', 'high_value', 'low_value', 'volume', 'return_1d']]

df_6m['intraday_volatility'] = (df_6m['high_value'] - df_6m['low_value']) / df_6m['open_value']
df['return_6m'] = df.groupby('ticker')['close_value'].transform(lambda x: (x - x.shift(126)) / x.shift(126))
return_6m = df.groupby('ticker')['return_6m'].last().reset_index()

summary = df_6m.groupby('ticker').agg({
    'return_1d': ['mean', 'std'],
    'intraday_volatility': 'mean',
    'volume': ['mean', 'std']
}).reset_index()

summary.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in summary.columns]
summary = summary.rename(columns={'ticker_': 'ticker'})
summary = summary.merge(return_6m, on='ticker', how='left')
summary.set_index('ticker', inplace=True)

In [ ]:
summary

In [ ]:
Q1 = summary[['return_1d_mean', 'return_1d_std', 'intraday_volatility_mean',
       'volume_mean', 'volume_std', 'return_6m']].quantile(0.25)
Q3 = summary[['return_1d_mean', 'return_1d_std', 'intraday_volatility_mean',
       'volume_mean', 'volume_std', 'return_6m']].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

filtered_df = summary[
    (summary['return_1d_mean'] >= lower_bound['return_1d_mean']) &
    (summary['return_1d_mean'] <= upper_bound['return_1d_mean']) &
    (summary['return_1d_std'] >= lower_bound['return_1d_std']) &
    (summary['return_1d_std'] <= upper_bound['return_1d_std']) &
    (summary['intraday_volatility_mean'] >= lower_bound['intraday_volatility_mean']) &
    (summary['intraday_volatility_mean'] <= upper_bound['intraday_volatility_mean']) &
    (summary['volume_mean'] >= lower_bound['volume_mean']) &
    (summary['volume_mean'] <= upper_bound['volume_mean']) &
    (summary['volume_std'] >= lower_bound['volume_std']) &
    (summary['volume_std'] <= upper_bound['volume_std']) &
    (summary['return_6m'] >= lower_bound['return_6m']) &
    (summary['return_6m'] <= upper_bound['return_6m'])
]

print(f"Rows removed: {len(summary) - len(filtered_df)}")

In [ ]:
filtered_df.head()

In [ ]:
std_scaler = StandardScaler()
scaled_features = std_scaler.fit_transform(filtered_df)

## 🤖 Step 3: Model Development

In [ ]:
inertia = []
silhouette_scores = []
k_range = range(1, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=12)
    km.fit(scaled_features)
    inertia.append(km.inertia_)

plt.figure(figsize=(12, 8))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow Method For Optimal K')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.grid()
plt.show()

## 📈 Step 4: Model Evaluation

In [ ]:
k = 3
km = KMeans(n_clusters=k, random_state=12)
labels = km.fit_predict(scaled_features)
filtered_df.loc[:, 'cluster'] = labels
sc_km = silhouette_score(scaled_features, labels)
print(f'Silhouette score for K-Means: {sc_km}')

In [ ]:
filtered_df.head()

In [ ]:
pca = PCA(n_components=3)
pca_data = pca.fit_transform(scaled_features)
pca_df = pd.DataFrame(data=pca_data, columns=['PC1', 'PC2', 'PC3'])
pca_df['clusters'] = labels

fig = px.scatter_3d(
    pca_df, x='PC1', y='PC2', z='PC3',
    color=pca_df['clusters'].astype(str),
    title='3D PCA Visualization of K-Means Clusters',
    labels={'PC1': 'PC 1', 'PC2': 'PC 2', 'PC3': 'PC 3'},
    opacity=0.8
)

fig.show()

#### Analysis

In [ ]:
cluster_summary = filtered_df.groupby('cluster').mean(numeric_only=True)
cluster_summary.head()

Cluster 1 → safest companies
Low volatility, good return → ideal for long-term investors

Cluster 2 → solid, but more active stocks
Acceptable for long-term, but more monitoring needed

Cluster 0 → avoid
Losing value and unstable

In [ ]:
centroids = km.cluster_centers_
centroids

In [ ]:
centroid_1 = centroids[1]
cluster_1_data = scaled_features[filtered_df['cluster'] == 1]
cluster_1_tickers = filtered_df[filtered_df['cluster'] == 1].index.values
cluster_1_tickers

In [ ]:
closest_idx, _  = pairwise_distances_argmin_min([centroid_1], cluster_1_data)
representative_ticker = cluster_1_tickers[closest_idx]
print("Closest company to centroid of cluster 1:", representative_ticker)

In our case the centroid is Cognizant Technology Solutions Corp (CTSH). Now we can use values of this company for building our classification model.

---

# Classification Model

##  🗄️ Step 1: Load and Explore Data

In [ ]:
import pandas as pd
from sqlalchemy import create_engine
from stock_chatbot.config import get_db_url

# Connection details come from environment variables (see .env.example)
db_url = get_db_url()

query = '''
SELECT *
FROM features;
'''

engine = create_engine(db_url, echo=True)
df_alch_2 = pd.read_sql_query(query, engine)

In [ ]:
df_alch_2.to_csv('features.csv', index=False)

## 📊 Step 2: Preprocessing

In [ ]:
import pandas as pd

df_c = pd.read_csv("features.csv", parse_dates=["date_value"])
df_c = df_c.sort_values(["company_prefix", "date_value"])

df_c.head()

In [ ]:
# Filter to CTSH only, then sort by date_value
df_ctsh = df_c[df_c["company_prefix"] == "CTSH"].sort_values("date_value").reset_index(drop=True)

# Quick check
print(df_ctsh.shape)
df_ctsh.head()

In [ ]:
# List of macro columns to forward fill
macro_cols = [
    "gdp_growth",
    "unemployment_rate",
    "inflation_cpi",
    "interest_rate_fed_funds",
    "10_year_treasury_yield",
    "stock_market_volatility_vix_index",
    "retail_sales_data_excluding_food_services"
]

# Apply forward/backward fill down each column
df_ctsh[macro_cols] = df_ctsh[macro_cols].ffill().bfill()

# Quick check to ensure no more gaps in those columns
print(df_ctsh[macro_cols].isnull().sum())
df_ctsh.head()

In [ ]:
# Drop rows where any price/volume field is missing
price_cols = ["open_value", "high_value", "low_value", "close_value", "volume"]

# Check how many missing values in these columns
print(df_ctsh[price_cols].isnull().sum())
df_ctsh.shape

### Feature Engineering:

In [ ]:
import numpy as np

# 126‐day Moving Average (MA_126)
df_ctsh['ma_126'] = df_ctsh['close_value'].rolling(window=126).mean()

# 126‐day Volatility (std dev of close)
df_ctsh['volatility_126'] = df_ctsh['close_value'].rolling(window=126).std()

# Percent change from yesterday’s close to today’s close
df_ctsh['return_1d'] = df_ctsh['close_value'].pct_change(periods=1) * 100

# Difference between today’s close and the close 126 days ago
df_ctsh['momentum_126'] = df_ctsh['close_value'] - df_ctsh['close_value'].shift(126)

# 126‐day Rate of Change (ROC_126) in percent
df_ctsh['roc_126'] = df_ctsh['close_value'].pct_change(periods=126) * 100

# Bollinger Bands (20-day MA ± 2×STD) and Bandwidth
ma_20 = df_ctsh['close_value'].rolling(window=20).mean()
std_20 = df_ctsh['close_value'].rolling(window=20).std()

df_ctsh['bollinger_upper']  = ma_20 + 2 * std_20
df_ctsh['bollinger_lower']  = ma_20 - 2 * std_20
df_ctsh['bollinger_bandwidth'] = (df_ctsh['bollinger_upper'] - df_ctsh['bollinger_lower']) / ma_20

# 14‐day RSI (Relative Strength Index)
delta = df_ctsh['close_value'].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()
rs = avg_gain / avg_loss
df_ctsh['rsi_14'] = 100 - (100 / (1 + rs))

# Moving-Average Crossover (20d vs 50d)
ma_50 = df_ctsh['close_value'].rolling(window=50).mean()
df_ctsh['ma_crossover_20_50'] = (ma_20 > ma_50).astype(int)

# 126‐day Sharpe Ratio (mean return / std dev of return)
mean_return_126 = df_ctsh['return_1d'].rolling(window=126).mean()
std_return_126 = df_ctsh['return_1d'].rolling(window=126).std()
df_ctsh['sharpe_126'] = mean_return_126 / std_return_126

# Volume Change (1-day)
df_ctsh['volume_change_1d'] = df_ctsh['volume'].pct_change(periods=1) * 100

# Drop the rows where any of these rolling features are still NaN
rolling_cols = ['ma_126', 'volatility_126', 'roc_126', 'rsi_14', 'sharpe_126']
df_ctsh = df_ctsh.dropna(subset=rolling_cols).reset_index(drop=True)

# Quick check to confirm no missing values in rolling features
print(df_ctsh[rolling_cols].isnull().sum())

df_ctsh.head()

In [ ]:
rolling_cols = [
    'ma_126', 'volatility_126', 'return_1d', 'momentum_126',
    'roc_126', 'bollinger_upper', 'bollinger_lower',
    'bollinger_bandwidth', 'rsi_14', 'ma_crossover_20_50',
    'volume_change_1d', 'sharpe_126'
]
df_ctsh = df_ctsh.dropna(subset=rolling_cols).reset_index(drop=True)

In [ ]:
# Create the 6-month return target (return_6m)
# Calculate return over the next 126 trading days (approximately 6 months)
df_ctsh['return_6m'] = (df_ctsh['close_value'].shift(-126) - df_ctsh['close_value']) / df_ctsh['close_value'] * 100

### Outlier Handling:

In [ ]:
import matplotlib.pyplot as plt

# List of key features to inspect for outliers
features_to_plot = [
    'return_6m',
    'ma_126',
    'volatility_126',
    'rsi_14',
    'sharpe_126'
]

# For each feature, draw a box plot
for feature in features_to_plot:
    plt.figure(figsize=(6, 3))
    plt.boxplot(df_ctsh[feature].dropna(), vert=True, patch_artist=True)
    plt.title(f"Boxplot of {feature}")
    plt.ylabel(feature)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()

In [ ]:
# Winsorize (Cap) at the IQR fences
Q1 = df_ctsh['return_6m'].quantile(0.25)
Q3 = df_ctsh['return_6m'].quantile(0.75)
IQR = Q3 - Q1
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

df_ctsh['return_6m_capped'] = df_ctsh['return_6m'].clip(lower_fence, upper_fence)

import matplotlib.pyplot as plt

plt.figure(figsize=(6, 3))
plt.boxplot(df_ctsh['return_6m_capped'].dropna(), vert=True, patch_artist=True)
plt.title("Boxplot of return_6m (capped)")
plt.ylabel("return_6m_capped")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 1) Define classification buckets for the capped return_6m_capped (percentages)
def classify_6m_capped(val):
    if val is None or np.isnan(val):
        return np.nan
    if val >= 30:
        return 'very_high'
    elif val >= 15:
        return 'high'
    elif val >= -5:
        return 'no_change'
    elif val >= -20:
        return 'low'
    else:
        return 'very_low'

# 2) Apply classification on the capped values instead of the original
df_ctsh['price_label'] = df_ctsh['return_6m_capped'].apply(classify_6m_capped)

# 3) Drop rows where price_label is still NaN (e.g., from any remaining NaNs)
df_ctsh = df_ctsh.dropna(subset=['price_label']).reset_index(drop=True)

# 4) Verify the new class distribution
print(df_ctsh['price_label'].value_counts(normalize=True))

df_ctsh.head()

In [ ]:
# 1) Compute the IQR‐based fences for rsi_14
Q1_rsi = df_ctsh['rsi_14'].quantile(0.25)
Q3_rsi = df_ctsh['rsi_14'].quantile(0.75)
IQR_rsi = Q3_rsi - Q1_rsi
lower_rsi = Q1_rsi - 1.5 * IQR_rsi
upper_rsi = Q3_rsi + 1.5 * IQR_rsi

# Create a new capped column
df_ctsh['rsi_14_capped'] = df_ctsh['rsi_14'].clip(lower_rsi, upper_rsi)

# 2) Compute the IQR‐based fences for sharpe_126
Q1_sh = df_ctsh['sharpe_126'].quantile(0.25)
Q3_sh = df_ctsh['sharpe_126'].quantile(0.75)
IQR_sh = Q3_sh - Q1_sh
lower_sh = Q1_sh - 1.5 * IQR_sh
upper_sh = Q3_sh + 1.5 * IQR_sh

# Create a new capped column
df_ctsh['sharpe_126_capped'] = df_ctsh['sharpe_126'].clip(lower_sh, upper_sh)

# 3) Compute the IQR‐based fences for volatility_126
Q1_vol = df_ctsh['volatility_126'].quantile(0.25)
Q3_vol = df_ctsh['volatility_126'].quantile(0.75)
IQR_vol = Q3_vol - Q1_vol
lower_vol = Q1_vol - 1.5 * IQR_vol
upper_vol = Q3_vol + 1.5 * IQR_vol

# Create a new capped column
df_ctsh['volatility_126_capped'] = df_ctsh['volatility_126'].clip(lower_vol, upper_vol)

In [ ]:
import matplotlib.pyplot as plt

# Boxplot for RSI (capped)
plt.figure(figsize=(6, 3))
plt.boxplot(df_ctsh['rsi_14_capped'].dropna(), vert=True, patch_artist=True)
plt.title("Boxplot of RSI-14 (capped)")
plt.ylabel("rsi_14_capped")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Boxplot for Sharpe Ratio (capped)
plt.figure(figsize=(6, 3))
plt.boxplot(df_ctsh['sharpe_126_capped'].dropna(), vert=True, patch_artist=True)
plt.title("Boxplot of Sharpe-126 (capped)")
plt.ylabel("sharpe_126_capped")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Boxplot for volatility_126 (capped)
plt.figure(figsize=(6, 3))
plt.boxplot(df_ctsh['volatility_126_capped'].dropna(), vert=True, patch_artist=True)
plt.title("Boxplot of volatility_126 (capped)")
plt.ylabel("'volatility_126_capped'")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
df_ctsh.info()

In [ ]:
df_ctsh = df_ctsh.drop(columns=['rsi_14', 'sharpe_126', 'volatility_126']).reset_index(drop=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Identify numeric columns for correlation (excluding feature_id and capped version of return_6m)
numeric_cols = df_ctsh.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [col for col in numeric_cols if col not in [
    'feature_id', 'return_6m_capped']]

# Ensure return_6m is included
if 'return_6m' not in numeric_cols:
    numeric_cols.append('return_6m')

# Compute correlation matrix for these columns
corr_matrix = df_ctsh[numeric_cols].corr()

# Plot heatmap using matplotlib
plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix, aspect='auto', cmap='viridis', interpolation='none')
plt.colorbar(label='Correlation Coefficient')

# Set ticks and labels
plt.xticks(ticks=np.arange(len(numeric_cols)), labels=numeric_cols, rotation=90)
plt.yticks(ticks=np.arange(len(numeric_cols)), labels=numeric_cols)

plt.title("Correlation Heatmap: Features vs. return_6m")
plt.tight_layout()
plt.show()

## 🤖 Step 3: Model Development

In [ ]:
from sklearn.preprocessing import LabelEncoder

features = [
    "volume",
    "transactions",
    "gdp_growth",
    "unemployment_rate",
    "inflation_cpi",
    "interest_rate_fed_funds",
    "10_year_treasury_yield",
    "stock_market_volatility_vix_index",
    "retail_sales_data_excluding_food_services",
    "volatility_126_capped",
    "return_1d",
    "momentum_126",
    "roc_126",
    "bollinger_bandwidth",
    "ma_crossover_20_50",
    "volume_change_1d",
    "rsi_14_capped",
    "sharpe_126_capped"
]

# Extract feature matrix X and target y
X = df_ctsh[features].copy()         # only these columns
y = df_ctsh["price_label"].copy()     # the 5‐class label

# Encode the text labels to integers 0–4
le = LabelEncoder()
y_encoded = le.fit_transform(y)
# Now, le.classes_ might be ['high','low','no_change','very_high','very_low']
# and y_encoded will be integers like 0,1,2,3,4 accordingly.

print("Label mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded)

print("Training shape:", X_train.shape, "Test shape:", X_test.shape)

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier

# Instantiate the model
model = XGBClassifier(
    objective="multi:softprob",   # multiclass probability output
    eval_metric="mlogloss",       # suitable multiclass loss
    random_state=42,              
    n_estimators=100,             
    max_depth=5,                  
    learning_rate=0.1,            
    subsample=0.8,                
    colsample_bytree=0.8,         
    reg_lambda=1                  
)

# Fit on the training data
model.fit(X_train, y_train)

# Predict on the test data
y_pred = model.predict(X_test)

## 📈 Step 4: Model Evaluation

In [ ]:
# Print a detailed classification report
print("Classification Report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

# Print the overall confusion matrix
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

In [ ]:
from sklearn.metrics import accuracy_score

print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}")

### Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    model, 
    X_train, 
    y_train, 
    cv=skf, 
    scoring="f1_macro"   # macro‐F1 because classes are somewhat imbalanced
)

print("5-Fold CV Macro-F1 scores:", cv_scores)
print("Mean CV Macro-F1:", cv_scores.mean())

### Feature Importance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract feature importances and sort them
imp_series = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)

# Plot as a horizontal bar chart
plt.figure(figsize=(8, 5))
imp_series.plot(kind="barh", title="XGBoost Feature Importance")
plt.xlabel("Importance")
plt.show()

## 🔧  Step 5: Hyperparameter Tuning (Classification)

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from xgboost import XGBClassifier

# 1) Build a pipeline that first oversamples, then fits XGBoost
pipe = Pipeline([
    ("ros", RandomOverSampler(random_state=42)),
    ("clf", XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        use_label_encoder=False,
        random_state=42
    )),
])

# 2) Specify the hyperparameter grid to search
param_grid = {
    "clf__max_depth":       [3, 5, 7],
    "clf__learning_rate":   [0.01, 0.05, 0.1],
    "clf__n_estimators":    [100, 200, 300],
    "clf__subsample":       [0.6, 0.8, 1.0],
    "clf__colsample_bytree":[0.6, 0.8, 1.0],
    "clf__reg_lambda":      [1, 10, 100],
}

# 3) Use stratified 3-fold CV optimizing macro-F1
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
gs = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1_macro",
    cv=cv,
    verbose=2,
    n_jobs=-1
)

# 4) Fit the search on the full dataset
gs.fit(X, y_encoded)

print("Best CV macro-F1:", gs.best_score_)
print("Best hyperparameters:", gs.best_params_)

### Why We Performed Hyperparameter Tuning?

While default XGBoost settings can deliver reasonable performance out of the box, they are rarely optimal for a specific dataset or task. Systematically searching over the key hyperparameters—such as tree depth, learning rate, number of trees, subsampling ratios, and regularization strengths—allows us to:

- **Maximize Generalization**: Find the sweet spot between underfitting (too simple) and overfitting (too complex).  
- **Optimize Our Objective**: Directly tune for our chosen metric (macro-F1) rather than relying on defaults that target log loss.  
- **Ensure Robustness**: Use cross‐validation to verify that performance gains hold across multiple folds, not just a single train/test split.  
- **Document Model Choices**: Produce a reproducible record of which hyperparameter combination we deployed, along with evidence that it outperforms both the defaults and naive baselines.  

By investing in this grid search, we have confidence that our final model configuration is data-driven, rather than guesswork, and delivers the best possible classification of six-month return buckets for our NASDAQ-100 universe.


### Why These Final Parameters Were Chosen?

After evaluating 729 candidate combinations via 3-fold stratified CV (scoring macro-F1), the best configuration was:

- **max_depth=7**: Provides sufficient tree complexity to capture non-linear interactions, without overfitting to noise.  
- **learning_rate=0.05**: A lower rate smooths learning, preventing large leaps and improving convergence stability.  
- **n_estimators=300**: A larger ensemble compensates for the smaller learning rate, allowing the model to refine its predictions across more boosting rounds.  
- **subsample=0.8 & colsample_bytree=0.6**: Randomly sampling 80 % of rows and 60 % of features at each tree reduces variance and builds diversification across trees.  
- **reg_lambda=10**: Stronger L2 regularization penalizes overly large leaf weights, further guarding against overfitting.  

This combination consistently achieved the highest macro-F1 (0.8775) in CV and balanced precision/recall across all five classes, making it our deployment candidate.



## ✅ Step 6: Testing and Backtesting (Classification)

In [ ]:
import pandas as pd
import numpy as np
from pandas.tseries.offsets import DateOffset
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

# If 'date_value' still exists as a column, move it to the index.
if "date_value" in df_ctsh.columns:
    df_ctsh = df_ctsh.set_index("date_value").sort_index()
else:
    df_ctsh = df_ctsh.sort_index()

# Confirm index type
print(f"Index dtype: {df_ctsh.index.dtype}")

In [ ]:
# 1) Grab the most recent timestamp from the DataFrame’s DateTimeIndex
last_date = df_ctsh.index[-1]

# 2) Build four “raw” cutoff dates spaced 18, 12, 6, and 0 months before last_date
raw_cutoffs = [
    last_date - DateOffset(months=6 * i)  # subtract 18m, then 12m, 6m, and 0m
    for i in (3, 2, 1, 0)
]

# 3) Align each raw cutoff to the nearest actual trading-day in the index
cutoffs = []
for pt in raw_cutoffs:
    # get_indexer returns the integer position of the closest index entry
    pos = df_ctsh.index.get_indexer([pt], method="nearest")[0]
    cutoffs.append(df_ctsh.index[pos])

# 4) Unpack the aligned timestamps for clarity
date_18m, date_12m, date_6m, date_0m = cutoffs

# 5) Print them out to verify the exact split points
print("Aligned cutoffs (18m, 12m, 6m, 0m):")
print(date_18m, date_12m, date_6m, date_0m)

In [ ]:
features = [
    "volume",
    "transactions",
    "gdp_growth",
    "unemployment_rate",
    "inflation_cpi",
    "interest_rate_fed_funds",
    "10_year_treasury_yield",
    "stock_market_volatility_vix_index",
    "retail_sales_data_excluding_food_services",
    "volatility_126_capped",
    "return_1d",
    "momentum_126",
    "roc_126",
    "bollinger_bandwidth",
    "ma_crossover_20_50",
    "volume_change_1d",
    "rsi_14_capped",
    "sharpe_126_capped"
]

# Confirm that all listed features actually exist in df_ctsh
missing = [f for f in features if f not in df_ctsh.columns]
if missing:
    print("WARNING: These feature columns are missing:", missing)

In [ ]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

def evaluate_split(train_end, test_start, split_name):
    # 1) Slice out train/test by date
    train_df = df_ctsh.loc[:train_end].dropna(subset=["price_label"])
    test_df  = df_ctsh.loc[test_start:].dropna(subset=["price_label"])

    # 2) Extract X & y
    X_train, y_train = train_df[features], train_df["price_label"]
    X_test,  y_test  = test_df[features],  test_df["price_label"]

    # 3) Encode labels
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    y_test_enc  = le.transform(y_test)

    # 4) Oversample minority classes
    ros = RandomOverSampler(random_state=42)
    X_res, y_res = ros.fit_resample(X_train, y_train_enc)
    print(f"After oversampling, train class distribution: {Counter(y_res)}")

    # 5) Train XGBoost with tuned hyperparameters
    # Best hyperparameters: {'clf__colsample_bytree': 0.6, 'clf__learning_rate': 0.05, 'clf__max_depth': 7, 'clf__n_estimators': 300, 'clf__reg_lambda': 10, 'clf__subsample': 0.8}
    model = XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_estimators=300,      # from GridSearchCV
        max_depth=7,           
        learning_rate=0.05,    
        subsample=0.8,         
        colsample_bytree=0.6,  
        reg_lambda=10          
    )
    model.fit(X_res, y_res)

    # 6) Predict & evaluate
    y_pred_enc = model.predict(X_test)
    acc = accuracy_score(y_test_enc, y_pred_enc)

    print(f"--- {split_name} ---")
    print(f"Train period: {train_df.index[0].date()} → {train_df.index[-1].date()}")
    print(f"Test  period: {test_df.index[0].date()} → {test_df.index[-1].date()}")
    print(f"Test accuracy: {acc:.4f}")
    print("Classification Report:")
    print(classification_report(y_test_enc, y_pred_enc, zero_division=0))
    print()

    return {"split": split_name, "accuracy": acc}

### Majority-Class Baseline

Before diving into our XGBoost backtest, it’s important to establish a **baseline**—in our case, a simple “always predict **no_change**” rule. This baseline serves two key purposes:

1. **Sanity Check**  
   - Ensures our pipeline (feature engineering, splitting, metrics) is wired up correctly.  
   - If even this trivial rule can’t run, something is wrong upstream.

2. **Performance Floor**  
   - “No_change” accounts for the majority of six-month outcomes (~50 %), so whatever accuracy a naive predictor achieves will be our floor.  
   - Any model we build must **beat** this floor to demonstrate real predictive value.

By comparing our XGBoost model against this naive baseline, we can quantify how much genuine signal we’re extracting—rather than merely echoing the dataset’s natural class imbalance.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Helper to run the majority‐class baseline on one split
def majority_baseline_split(train_end, test_start, split_name):
    # Slice train/test by our cutoffs
    train_df = df_ctsh.loc[:train_end].dropna(subset=["price_label"])
    test_df = df_ctsh.loc[test_start:].dropna(subset=["price_label"])
    
    # Find the most frequent label in the training slice
    most_common = train_df["price_label"].mode()[0]
    
    # Predict it for every test row
    y_true = test_df["price_label"]
    y_pred = [most_common] * len(y_true)
    
    # Report
    acc = accuracy_score(y_true, y_pred)
    print(f"--- Baseline {split_name} (always '{most_common}') ---")
    print(f"Test accuracy: {acc:.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    print()
    return acc

# Run it on the three splits
baseline_scores = []
baseline_scores.append(majority_baseline_split(date_18m, date_18m, "18→12m"))
baseline_scores.append(majority_baseline_split(date_12m, date_12m, "12→6m"))
baseline_scores.append(majority_baseline_split(date_6m,  date_6m,  "6→0m"))

print("Average baseline accuracy:", sum(baseline_scores)/len(baseline_scores))

In [ ]:
results = []
results.append(evaluate_split(date_18m, date_18m, "Split 1: 18 → 12 months"))
results.append(evaluate_split(date_12m, date_12m, "Split 2: 12 → 6 months"))
results.append(evaluate_split(date_6m,  date_6m,  "Split 3: 6 → 0 months"))

print("Average backtest accuracy:", np.mean([r["accuracy"] for r in results]))

### Why Our Model Isn’t Yet Beating the Baseline

After running both the baseline and the full XGBoost backtest, we observed:

- **Baseline accuracy** ≈ 70 % (always “no_change”)  
- **XGBoost accuracy** ≈ 61 %

A few factors that likely explain this underperformance:

1. **Outlier Removal in Clustering**  
   - Earlier, when clustering companies by risk, we **capped** and **dropped** returns beyond the 25 % – 75 % range to isolate “safe” firms.  
   - This process likely **removed** many of the large-move examples (the non-“no_change” cases) from our training set, making them rare or absent.

2. **Severe Class Imbalance**  
   - With most extreme returns filtered out, our model sees very few “low,” “high,” or “very_high” examples. So it struggles to learn their patterns and defaults to “no_change.”

3. **Feature Signal Strength**  
   - The engineered indicators (MA, RSI, Sharpe, etc.) may not carry enough information to distinguish the modest number of tail events once outliers have been trimmed.

#### Next Steps

- **Reintroduce some tail data** (e.g. widen the outlier caps) so the model sees more examples of true moves.   
- **Enhance features** targeting tail behavior (e.g. shorter-term momentum, macro surprises, mid-term indicators).  
- **Explore different model families**, such as:
  - Classical time-series models (ARIMA/SARIMA) for direct 6-month forecasts  
  - Sequence models (LSTM, TCN, Transformers) to learn patterns from raw multivariate series  
  - Alternative tree-ensembles (LightGBM, CatBoost) with native categorical handling and different regularization  
- **Ensemble approaches** (stacking or blending XGBoost with simpler models) to capture complementary signals  
- **Final hold-out validation** on a truly unseen six-month period to confirm production readiness  

Until we address these factors, a majority-class baseline remains hard to beat, but understanding this gap guides us toward more effective model refinements.  

In [ ]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

# Save the model and label encoder to disk
joblib.dump(model, "../models/xgb_ctsh_model.pkl")
joblib.dump(le, "../models/label_encoder_ctsh.pkl")

print("Model and encoder saved!")

### Applying The Trained Model on The Safest Companies:

In [ ]:
import pandas as pd
import joblib
from sqlalchemy import create_engine
from stock_chatbot.config import get_db_url

# Create the SQLAlchemy engine
db_url = get_db_url()  # from environment variables (see .env.example)
engine = create_engine(db_url)

# Load the final_classification table (safest companies table with engineered features)
df_to_score = pd.read_sql_table(
    table_name="final_classification",
    con=engine,
    schema="public"
)

# Ensure any datetime columns are parsed
df_to_score["date_value"] = pd.to_datetime(df_to_score["date_value"])

# Define the feature columns exactly as used in training
features = [
    "volume", "transactions", "gdp_growth", "unemployment_rate",
    "inflation_cpi", "interest_rate_fed_funds", "10_year_treasury_yield",
    "stock_market_volatility_vix_index", "retail_sales_data_excluding_food_services",
    "volatility_126_capped", "return_1d", "momentum_126", "roc_126",
    "bollinger_bandwidth", "ma_crossover_20_50", "volume_change_1d",
    "rsi_14_capped", "sharpe_126_capped"
]

X_new = df_to_score[features]

# Load the saved model and label encoder
model = joblib.load("../models/xgb_ctsh_model.pkl")
le = joblib.load("../models/label_encoder_ctsh.pkl")

# Predict encoded labels and map back to text
y_pred_enc = model.predict(X_new)
y_pred = le.inverse_transform(y_pred_enc)

# Attach the predictions to the DataFrame
df_to_score["predicted_price_label"] = y_pred

# Inspect a sample
print(df_to_score[["company_prefix","date_value","predicted_price_label"]].head())

## 🗃️  Step 7: Create a Database Table with Your Final Results

### Database Table (With Classification Results)

In [ ]:
# Write the scored DataFrame back to the warehouse
df_to_score.to_sql(name="final_classification_scored", con=engine, if_exists="replace", index=False)

### Database Table (With Clustering Results)

In [ ]:
result  = summary.copy()

result['cluster'] = result.index.map(filtered_df['cluster'])

def explain_category(cluster):
    if pd.isna(cluster):
        return 'outlier'
    elif cluster == 0:
        return 'avoid'
    elif cluster == 1:
        return 'safest'
    elif cluster == 2:
        return 'solid'
    else:
        return 'Unknown'

result['category'] = result['cluster'].apply(explain_category)

In [ ]:
result.to_sql(name = 'clustering_results', con =engine, if_exists='replace', index=True)

## 🚀 Step 8: Preparation for Deployment

In [ ]:
os.makedirs("../models", exist_ok=True)
# Save model(s) for deployment
joblib.dump(model, "../models/xgb_ctsh_model.pkl")
joblib.dump(le, "../models/label_encoder_ctsh.pkl")

joblib.dump(km, "../models/kmeans_model.pkl")
joblib.dump(std_scaler, "../models/standard_scaler.pkl")

print("Models, encoder and scaler saved!")